In [10]:
#=====IMPORTS======
import sys
from pathlib import Path

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

sys.path.append(str(Path("../src").resolve()))

from common_utils import (
    evaluate_predictions,
    load_split_data,
    print_evaluation_results,
    save_metrics,
    save_sklearn_model,
)

In [11]:
#=====PATHS======
TRAIN_CSV_PATH = "dataset/processed_data/train.csv"
VALIDATION_CSV_PATH = "dataset/processed_data/validation.csv"
TEST_CSV_PATH = "dataset/processed_data/test.csv"

OUTPUT_DIRECTORY = "outputs/tfidf_logreg"
MODEL_FILE_NAME = "tfidf_logreg_baseline.joblib"

In [12]:
train_dataframe, validation_dataframe, test_dataframe = load_split_data(
    train_csv_path=TRAIN_CSV_PATH,
    validation_csv_path=VALIDATION_CSV_PATH,
    test_csv_path=TEST_CSV_PATH,
)

print("Train shape:", train_dataframe.shape)
print("Validation shape:", validation_dataframe.shape)
print("Test shape:", test_dataframe.shape)

print("\nTraining label distribution:")
print(train_dataframe["generated"].value_counts())

Train shape: (371759, 2)
Validation shape: (46470, 2)
Test shape: (46470, 2)

Training label distribution:
generated
0    227934
1    143825
Name: count, dtype: int64


In [13]:
tfidf_logreg_pipeline = Pipeline(
    steps=[
        (
            "tfidf_vectorizer",
            TfidfVectorizer(
                lowercase=True,
                stop_words="english",
                ngram_range=(1, 2),
                max_features=20000,
                min_df=2,
                max_df=0.95,
                sublinear_tf=True,
            ),
        ),
        (
            "logistic_regression_classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

tfidf_logreg_pipeline

Pipeline(steps=[('tfidf_vectorizer',
                 TfidfVectorizer(max_df=0.95, max_features=20000, min_df=2,
                                 ngram_range=(1, 2), stop_words='english',
                                 sublinear_tf=True)),
                ('logistic_regression_classifier',
                 LogisticRegression(class_weight='balanced', max_iter=2000,
                                    random_state=42))])

In [14]:
tfidf_logreg_pipeline.fit(
    train_dataframe["text"],
    train_dataframe["generated"],
)

Pipeline(steps=[('tfidf_vectorizer',
                 TfidfVectorizer(max_df=0.95, max_features=20000, min_df=2,
                                 ngram_range=(1, 2), stop_words='english',
                                 sublinear_tf=True)),
                ('logistic_regression_classifier',
                 LogisticRegression(class_weight='balanced', max_iter=2000,
                                    random_state=42))])

In [15]:
validation_predicted_labels = tfidf_logreg_pipeline.predict(validation_dataframe["text"])
validation_predicted_probabilities = tfidf_logreg_pipeline.predict_proba(validation_dataframe["text"])[:, 1]

validation_results = evaluate_predictions(
    true_labels=validation_dataframe["generated"],
    predicted_labels=validation_predicted_labels,
    predicted_probabilities=validation_predicted_probabilities,
)

print_evaluation_results("TF-IDF + Logistic Regression Validation", validation_results)


=== TF-IDF + Logistic Regression Validation Results ===
Accuracy : 0.9960
Precision: 0.9962
Recall   : 0.9934
F1-score : 0.9948

Confusion Matrix:
[[28423, 69], [119, 17859]]

Classification Report:
              precision    recall  f1-score   support

       Human       1.00      1.00      1.00     28492
          AI       1.00      0.99      0.99     17978

    accuracy                           1.00     46470
   macro avg       1.00      1.00      1.00     46470
weighted avg       1.00      1.00      1.00     46470



In [16]:
test_predicted_labels = tfidf_logreg_pipeline.predict(test_dataframe["text"])
test_predicted_probabilities = tfidf_logreg_pipeline.predict_proba(test_dataframe["text"])[:, 1]

test_results = evaluate_predictions(
    true_labels=test_dataframe["generated"],
    predicted_labels=test_predicted_labels,
    predicted_probabilities=test_predicted_probabilities,
)

print_evaluation_results("TF-IDF + Logistic Regression Test", test_results)


=== TF-IDF + Logistic Regression Test Results ===
Accuracy : 0.9959
Precision: 0.9957
Recall   : 0.9938
F1-score : 0.9947

Confusion Matrix:
[[28414, 78], [111, 17867]]

Classification Report:
              precision    recall  f1-score   support

       Human       1.00      1.00      1.00     28492
          AI       1.00      0.99      0.99     17978

    accuracy                           1.00     46470
   macro avg       1.00      1.00      1.00     46470
weighted avg       1.00      1.00      1.00     46470



In [17]:
save_sklearn_model(
    model_object=tfidf_logreg_pipeline,
    output_directory=OUTPUT_DIRECTORY,
    file_name=MODEL_FILE_NAME,
)

save_metrics(
    evaluation_results=validation_results,
    output_directory=OUTPUT_DIRECTORY,
    file_name="validation_metrics.json",
)

save_metrics(
    evaluation_results=test_results,
    output_directory=OUTPUT_DIRECTORY,
    file_name="test_metrics.json",
)

Saved model to: outputs/tfidf_logreg/tfidf_logreg_baseline.joblib
Saved metrics to: outputs/tfidf_logreg/validation_metrics.json
Saved metrics to: outputs/tfidf_logreg/test_metrics.json
